# Compute Tidy DataFrame

## construct_scores.csv

In [ ]:
import os
import json

import pandas as pd

from typing import Literal
from pathlib import Path
from llm_audit import BASE_DIR
from llm_audit.util import construct_output_dir_label, get_supported_languages
from llm_audit.datasets.util import (
    get_dataset_by_label,
    complete_results_exist_across_models,
    get_dataset_label_class_map,
    format_raw_score_int_str_cast,
    contains_polarity_or_inverted_variable,
)

temperature: float = 1.0
runs: int = 10
seed: int = 42
model_selection_file: str = "final_complete.json"
output_dir_prefix_tags: list[str] = ["final", "final-authsys", "final-reverse"]
experiment_ablations: list[str] = ["default", "authsys", "reverse"]
assert len(output_dir_prefix_tags) == len(experiment_ablations), "missing prefix or experiment run label"
experiment_type_labels = ["open_question", "closed_question"]

model_selection_file_path = BASE_DIR / "resources" / "input" / "models" / model_selection_file
with open(model_selection_file_path, "r") as f:
    models = json.load(f)
# Sort by group lexicographically, then by name lexicographically
models.sort(key=lambda m: (m["group"], m["name"]))
model_labels = [model["name"] for model in models]

languages = get_supported_languages()
dataset_labels: list[str] = list(get_dataset_label_class_map().keys())

data = []
for experiment_type_label in experiment_type_labels:
    for dataset_label in dataset_labels:
        dataset = get_dataset_by_label(dataset_label=dataset_label)

        for language in languages:
            target_dir_labels: list[str] = []
            for output_dir_prefix_tag in output_dir_prefix_tags:
                _target_dir_label = construct_output_dir_label(
                    output_dir_prefix_tag=output_dir_prefix_tag,
                    language=language,
                    temperature=temperature,
                    runs=runs,
                    seed=seed,
                )
                target_dir_labels.append(_target_dir_label)

            target_dir_paths: list[Path] = []
            for target_dir_label in target_dir_labels:
                _target_dir_path = (
                    BASE_DIR / "resources" / "output" / target_dir_label / dataset_label / experiment_type_label
                )
                target_dir_paths.append(_target_dir_path)

            for target_dir_label, experiment_ablation in zip(target_dir_labels, experiment_ablations):
                # Assumption: reverse and authsys only en
                if not (experiment_ablation != "default" and language != "en"):
                    assert complete_results_exist_across_models(
                        model_names=model_labels,
                        experiment_output_dir_label=target_dir_label,
                        dataset_label=dataset_label,
                        experiment_type_label=experiment_type_label,
                        language=language,
                        verbose=True,
                    )

            dataset = get_dataset_by_label(dataset_label=dataset_label)
            ids: list[str] = [str(id) for id in dataset.get_ids(language=language)]

            for model_label in model_labels:
                for id in ids:
                    factor = dataset.get_factor(language=language, id=int(id))
                    reverse_scored_item: Literal["+", "-"] | None = dataset.get_polarity(
                        language=language,
                        id=int(id),
                        polarity_literal=contains_polarity_or_inverted_variable(dataset=dataset, language=language),
                    )
                    # default "+" poalarized / not "-" polarized / not reverse scored / not inverted
                    reverse_scored = 0 if reverse_scored_item is None or reverse_scored_item == "+" else 1

                    for target_dir_path, experiment_ablation in zip(target_dir_paths, experiment_ablations):
                        # Assumption: reverse and authsys only en
                        if experiment_ablation != "default" and language != "en":
                            continue

                        results_file = os.path.join(target_dir_path, id, model_label, "results.json")
                        with open(results_file, "r") as f:
                            results = json.load(f)
                        for run_idx, run in enumerate(results):
                            raw_score: str | None = run["response_value"]

                            refusal = 1 if raw_score is None else 0
                            adjusted_dir_aligned_normalized_score = None

                            if raw_score is not None:
                                adjusted_score: int = dataset.get_adjusted_score(
                                    language=language,
                                    id=int(id),
                                    raw_score=int(raw_score),
                                )
                                # Flip if needed to align with min to max disagree to agree dir
                                adjusted_dir_aligned_score = None
                                if not dataset.is_ordered_disagree_to_agree_asc():
                                    adjusted_dir_aligned_score = int(
                                        dataset.map_inverted_score[
                                            format_raw_score_int_str_cast(
                                                dataset=dataset,
                                                raw_score=adjusted_score,
                                            )
                                        ]
                                    )
                                else:
                                    adjusted_dir_aligned_score = int(adjusted_score)
                                # Normalize such that min, neutral, max -> -1, neutral threshold, +1
                                adjusted_dir_aligned_normalized_score = None
                                l_border = min(map(int, dataset.get_scale_items()))
                                r_border = max(map(int, dataset.get_scale_items()))
                                center = dataset.get_agreement_discriminator_threshold()
                                _x = adjusted_dir_aligned_score
                                if adjusted_dir_aligned_score <= center:
                                    adjusted_dir_aligned_normalized_score = (_x - center) / (center - l_border)
                                else:
                                    adjusted_dir_aligned_normalized_score = (_x - center) / (r_border - center)

                            # Append entry
                            data.append(
                                {
                                    "model": model_label,
                                    "dataset": dataset_label,
                                    "statement_id": int(id),
                                    "repetition_id": run_idx,  # 0 indexed [0, runs)
                                    "language": language,
                                    "experiment_type": experiment_type_label,
                                    "experiment_ablation": experiment_ablation,
                                    "score": adjusted_dir_aligned_normalized_score,
                                    "refusal": refusal,
                                    "reverse_scored": reverse_scored,
                                    "factor": factor,
                                    "auth": (
                                        None
                                        if adjusted_dir_aligned_normalized_score is None
                                        else 1
                                        if adjusted_dir_aligned_normalized_score > 0
                                        else 0
                                    ),
                                }
                            )

df = pd.DataFrame(data)
output_path = BASE_DIR / "eval" / "data" / "tidy" / "construct_scores.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

## vignette_scores.csv

In [ ]:
import os
import json

import pandas as pd

from pathlib import Path
from llm_audit import BASE_DIR
from llm_audit.util import construct_output_dir_label, get_supported_languages
from llm_audit.datasets.util import (
    get_dataset_by_label,
    complete_results_exist_across_models,
)

temperature: float = 1.0
runs: int = 10
vignettes_per_statement: int = 10
seed: int = 42
model_selection_file: str = "final_complete.json"
output_dir_prefix_tags: list[str] = [
    "final-vignette-rwa3d",
    "final-authsys-vignette-rwa3d",
]
experiment_ablations: list[str] = ["default", "authsys"]
assert len(output_dir_prefix_tags) == len(experiment_ablations), "missing prefix or experiment run label"

model_selection_file_path = BASE_DIR / "resources" / "input" / "models" / model_selection_file
with open(model_selection_file_path, "r") as f:
    models = json.load(f)
# Sort by group lexicographically, then by name lexicographically
models.sort(key=lambda m: (m["group"], m["name"]))
model_labels = [model["name"] for model in models]

# languages = get_supported_languages()  # Not doing multilingual stuff anymore
languages = ["en"]

data = []

experiment_type_label = "case_vignette"
dataset_label = "VignetteRWA3D"
dataset_vignette_rwa3d = get_dataset_by_label(dataset_label=dataset_label)
dataset_rwa3d = get_dataset_by_label(dataset_label="RWA3D")

for language in languages:
    target_dir_labels: list[str] = []
    for output_dir_prefix_tag in output_dir_prefix_tags:
        _target_dir_label = construct_output_dir_label(
            output_dir_prefix_tag=output_dir_prefix_tag,
            language=language,
            temperature=temperature,
            runs=runs,
            seed=seed,
        )
        target_dir_labels.append(_target_dir_label)

    target_dir_paths: list[Path] = []
    for target_dir_label in target_dir_labels:
        _target_dir_path = BASE_DIR / "resources" / "output" / target_dir_label / dataset_label / experiment_type_label
        target_dir_paths.append(_target_dir_path)

    for target_dir_label, experiment_ablation in zip(target_dir_labels, experiment_ablations):
        # Assumption: reverse and authsys only en
        if not (experiment_ablation != "default" and language != "en"):
            assert complete_results_exist_across_models(
                model_names=model_labels,
                experiment_output_dir_label=target_dir_label,
                dataset_label=dataset_label,
                experiment_type_label=experiment_type_label,
                language=language,
                verbose=True,
            )

    # RWA3D ids! (vignette-rwa3d ids would be flattened!)
    ids: list[str] = [str(id) for id in dataset_rwa3d.get_ids(language=language)]

    for model_label in model_labels:
        for id in ids:
            # Get factor from RWA3D dataset
            factor = dataset_rwa3d.get_factor(language=language, id=int(id))

            for target_dir_path, experiment_ablation in zip(target_dir_paths, experiment_ablations):
                # Assumption: authsys only en
                if experiment_ablation != "default" and language != "en":
                    continue

                results_file = os.path.join(target_dir_path, id, model_label, "results.json")
                with open(results_file, "r") as f:
                    results = json.load(f)

                # runs: int = 10 (repetitions)
                # vignettes_per_statement: int = 10
                assert len(results) == runs * vignettes_per_statement

                for run_idx, run in enumerate(results):
                    vignette_id = run_idx // 10
                    repetition_id = run_idx % 10

                    response_label: str | None = run["response_value"]
                    refusal = 1 if response_label is None else 0

                    # Append entry
                    data.append(
                        {
                            "model": model_label,
                            "dataset": dataset_label,
                            "language": language,
                            "statement_id": int(id),
                            "vignette_id": vignette_id,  # 0 indexed [0, vignettes_per_statement)
                            "repetition_id": repetition_id,  # 0 indexed [0, runs)
                            "experiment_ablation": experiment_ablation,
                            "response_label": response_label,
                            "refusal": refusal,
                            "factor": factor,
                            "auth": (
                                None
                                if response_label is None
                                else 1
                                if response_label in ["autho_medium", "autho_high"]
                                else 0
                            ),
                        }
                    )

df = pd.DataFrame(data)
output_path = BASE_DIR / "eval" / "data" / "tidy" / "vignette_scores.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)